# Buying-Intent Classifier — v14-Antiphony transferred to the Generic e-commerce corpus

**What this notebook is.** The v14-Antiphony architecture
(`Intention/Classifier/notebooks/improvement-nb/capstone-v14-antiphony.ipynb`),
the best-performing model on the PersuasionForGood donation corpus, applied
**unchanged in its architecture** to a different dialogue domain: 500
buyer–merchant e-commerce conversations labelled for `buying_intent`.

**Why.** The donation result answers "can this architecture learn *this*
task". It does not answer "is this a **dialogue-understanding pipeline**, or
a donation-corpus-specific model". Transferring the architecture to a corpus
that shares the *structure* (two asymmetric roles, one arguing and one
deciding; the label defined entirely by the deciding party's closing
position) but shares **no vocabulary, no domain, and no data source**
separates those two claims. Anything that transfers is a property of the
architecture; anything that collapses was a property of the donation data.

---

## The structural mapping

v14-Antiphony is built on one asymmetry: **one party argues, the other
decides, and the label is defined by what the deciding party does.** That
asymmetry is exactly what the e-commerce corpus has, with the roles renamed:

| PersuasionForGood | Generic e-commerce | role in the architecture |
|---|---|---|
| `[Persuader]` — solicits a donation | `[Merchant]` — sells | **argues** → cross-attention *key/value* |
| `[Persuadee]` — decides to give | `[Buyer]` — decides | **decides** → cross-attention *query*, GRU trajectory, pooled summary |
| `binary_label` ∈ {yes, no} | `buying_intent` ∈ {yes, no} | the target |

The CSV (`Buying_Intent_Label.csv`, built from `final.json`) writes Merchant
turns as `[Merchant]` and Buyer turns as `[Buyer]`, so **the model, collate,
and loss code below is identical in every computational respect to
v14-antiphony's** (only the marker regex and the two role-constant names
differ — they now read `[Merchant]`/`[Buyer]` and `ROLE_MERCHANT`/`ROLE_BUYER`,
denoting the same arguer/decider slots).**
The architecture is not re-implemented for this domain; it is re-used. That
is the point of the experiment.

---

## What changed from `capstone-v14-antiphony.ipynb`, and why

Four changes, all in data handling and evaluation. **The model class,
the loss, and the training loop are untouched.**

1. **No `modifier` head, no `modifier` column.** v14 was already binary-only
   (it forks v13-Solo). The generic corpus has no modifier field at all; it
   has `outcome_category` (4 values) instead, which is used **only** for
   stratifying the split and for slice-wise error analysis — never as a
   training target. The `modifier_*` plumbing that v14 still carried as
   dead weight (`modifier_id` columns, `batch.pop("modifier_labels")`) is
   kept as a constant zero column so the collate/train code stays literally
   identical, rather than being edited out.

2. **Augmentation is OFF** (`use_augmentation = False`). v14's augmentation
   plan oversamples by modifier class (`conditional` ×6, `deferred` ×2) to
   repair a 2.1 % minority class. This corpus is **52.4 / 47.6 balanced** —
   applying that plan would actively *introduce* the imbalance it was
   written to fix. The full EDA machinery is kept below, with its synonym
   and protected-word lists re-pointed at commerce vocabulary, so it can be
   switched on for a follow-up run without rewriting anything.

3. **A harder majority baseline.** Donation was 72.6 / 27.4 skewed, so a
   constant predictor scored 0.841 accuracy / 0.420 macro-F1. Here a
   constant predictor scores ≈ 0.52 accuracy / ≈ 0.34 macro-F1. **The
   headline numbers are not comparable across the two corpora** — only the
   *margin over each corpus's own baseline* is. Section 9 computes the
   baseline on this test split specifically.

4. **A last-turn ablation (new, §11).** The generic labels are derived from
   explicit decision moves ("lemme think, text u later" → `DEFERRED_CONSIDERATION`).
   If so, a model reading **only the Buyer's final turn** might match the
   full dual-stream model — which would mean the high F1 measures *label
   explicitness*, not dialogue understanding. §11 trains exactly that
   single-turn control on the same splits. **The gap between it and the
   full model is the quantity this notebook actually reports.**

5. **A controlled-skew stress test (new, §12).** Because this corpus is
   balanced, the class-balanced machinery inherited from v14 would otherwise
   never be *exercised* — leaving the cross-corpus comparison open to the
   objection that this domain was simply easier. §12 manufactures the skew
   (down to 10 % minority, harsher than donation's 31.8 %) and trains each
   ratio twice, with and without the imbalance handling, to show the
   technique working rather than merely being present.

Role markers are `[Merchant]` / `[Buyer]` rather than
`[Persuader]` / `[Persuadee]` — domain-accurate names for the same two
structural slots. Only the marker literals and two constant names change; no
tensor shape, mask convention, or code path differs.

---

## What each outcome would mean

| Full model | Last-turn ablation | Reading |
|---|---|---|
| high | **much lower** | The architecture is genuinely reading discourse; the dual-stream mechanism transfers as a *pipeline*. Strongest result. |
| high | **≈ equal** | The task is solvable from the closing turn alone. The pipeline transfers, but this corpus does not *test* dialogue understanding — a limitation to state plainly, not bury. |
| low | any | The architecture does not transfer off the donation corpus; the v14 result was domain-specific. Informative negative. |

For reference, v14-Antiphony on donation (test, n=153, tuned threshold):
`roberta-base` **0.859**, `deberta-v3-base` **0.860**, `todbert` **0.838**
macro-F1, against a **0.420** majority baseline.

## 0a. Running this on Kaggle

1. **Create the Dataset.** Kaggle → *Datasets* → *New Dataset* → upload
   `Buying_Intent_Label.csv`. Give it any name; the notebook does not depend on
   the slug.
2. **Attach it.** In this notebook: *+ Add Input* → *Datasets* → select it. It
   mounts read-only at `/kaggle/input/<your-slug>/Buying_Intent_Label.csv`.
3. **Run all.** §2 searches `/kaggle/input` **recursively** for a `*.csv` whose
   filename contains `buying_intent`, so no path needs editing regardless of
   the slug Kaggle assigns.
4. **Turn on the GPU** (*Settings → Accelerator → GPU T4*). On CPU this will
   not finish in a session.

Everything is written to `/kaggle/working`: checkpoints, per-section PNGs,
`results_summary_*.csv`, `skew_stress_*.csv`, `test_predictions_*.csv`, and a
consolidated `run_report_*.json`.

**Runtime budget on a single T4.** The three main encoders dominate, and
DeBERTa-v3 runs at `batch_size=2` (its disentangled attention builds extra
O(L²) tensors per layer, which the `[B*U, L]` turn-flattening multiplies).
If you are near the session limit, in order of what to cut:

- `CONFIG["run_skew_stress_test"] = False` — saves 8 runs
- trim `CONFIG["skew_ratios"]` to `[0.48, 0.15]` — saves 4
- drop `deberta-v3-base` from `CONFIG["encoders"]` — saves the slowest encoder

The dataset is small (500 dialogues, ≤ 32 turns each), so nothing here is
memory-bound apart from DeBERTa; gradient checkpointing is on unconditionally.

## 0b. Install / upgrade dependencies

In [ ]:
# =============================================================
# 0b. INSTALL / UPGRADE DEPENDENCIES (online mode)
# =============================================================
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.44.0,<4.47.0", "sentencepiece", "tqdm",
    "seaborn", "scikit-learn",
], check=True)
print("dependencies ok")

## 1. Imports & global config

In [ ]:
# =============================================================
# 1. IMPORTS & GLOBAL CONFIG
# =============================================================
import os
# Set BEFORE torch initializes its CUDA allocator -- mitigates the
# fragmentation-related OOM v6 hit on DeBERTa-v3. Unchanged from v14.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, json, glob, gc, time, warnings, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"torch version: {torch.__version__} (built for CUDA {torch.version.cuda})")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ---- CONFIG: the only block you should need to touch -----------------
CONFIG = {
    # Same fuzzy recursive search convention as v3-v14, so this works
    # unmodified whether the CSV is a Kaggle Dataset input or sits in
    # the repo next to this notebook.
    # KAGGLE: /kaggle/input is searched first and recursively, so a Dataset
    # added via "+ Add Input" is found wherever Kaggle mounts it
    # (/kaggle/input/<your-dataset-slug>/Buying_Intent_Label.csv) with no
    # path edit. The local entries are the fallback for running this
    # notebook straight from the repo.
    "search_roots": ["/kaggle/input", "/kaggle/working", "/workspace", "/data",
                     ".", "..", "../.."],

    # CHANGED from v14: the search token is "buying_intent", not "label".
    # v14 searched for *label*.csv, which on this repo would also match
    # the donation corpus's Manual_Label.csv and silently concatenate two
    # different tasks into one dataframe.
    "csv_must_contain": ["buying_intent"],

    "out_dir": "/kaggle/working" if os.path.isdir("/kaggle/working") else "./outputs",
    "run_tag": "generic-v14-antiphony",

    # CHANGED from v14: no "modifier" entry -- this corpus has no such
    # column. outcome_category is loaded separately, for stratification
    # and error analysis only; it is never a training target.
    "column_map": {
        "text": "dialogue_text",
        "binary_label": "binary_label",
    },
    "stratify_col": "outcome_category",   # analysis/split only, NOT a label
    # Descriptive columns carried for SLICE ANALYSIS ONLY (section 13).
    # They are never tokenized: column_map above routes only `text` to the
    # model. source_model and turn_count are deliberately NOT here and not
    # in the CSV -- source_model is an artifact of how the data was
    # generated, and turn_count is a length statistic that correlates with
    # outcome (dropouts are short) without being evidence of intent.
    # Neither is a property of the conversation a classifier should see.
    "analysis_cols": ["outcome_category", "archetype", "product", "category"],

    "binary_classes": ["no", "yes"],

    "encoders": [
        {"name": "roberta-base",    "hf_id": "roberta-base",              "batch_size": 8},
        {"name": "deberta-v3-base", "hf_id": "microsoft/deberta-v3-base", "lr": 5e-6,
         "batch_size": 2, "eval_batch_size": 4, "warmup_ratio": 0.10},
        {"name": "todbert",         "hf_id": "TODBERT/TOD-BERT-JNT-V1",   "batch_size": 8},
    ],

    # Measured on this corpus (see the audit cell in section 2):
    #   turns/dialogue: min 6, median 16, p95 28, max 32
    #   words/turn:     median 13, p95 31, max 118
    # So max_utterances=32 covers 100% of dialogues with no truncation,
    # and max_utt_len=64 tokens covers the p95 turn with full headroom.
    # Identical values to v14 -- they happen to fit both corpora.
    "max_utterances": 32,
    "max_utt_len": 64,

    # Antiphony architecture hyperparameters -- IDENTICAL to v14.
    # max_role_len=20: this corpus's per-role turn count maxes at 16,
    # so no role-stream truncation ever fires.
    "antiphony_stream_layers": 2,
    "antiphony_stream_heads": 8,
    "antiphony_cross_heads": 8,
    "antiphony_max_role_len": 20,

    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,

    "batch_size": 8,
    "eval_batch_size": 16,
    "num_epochs": 25,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "max_grad_norm": 1.0,
    "binary_loss_weight": 1.0,

    # ---- Augmentation: OFF for this corpus ---------------------------
    # v14's plan oversampled by modifier class to repair a 2.1% minority.
    # This corpus is 52.4/47.6 balanced, so that plan would CREATE
    # imbalance rather than fix it. Machinery kept (with commerce-domain
    # vocabulary) and gated off -- flip this to True for a follow-up run.
    "use_augmentation": False,
    "aug_alpha": 0.15,
    "aug_num_per_row": 1,          # uniform across BOTH classes when enabled
    "aug_buyer_bias_weight": 0.75,

    # ---- Loss: unchanged from v4/v13/v14 -----------------------------
    "cb_beta": 0.999,
    "focal_gamma": 2.0,
    "use_contrastive": True,
    "contrastive_temperature": 0.1,
    "contrastive_weight_binary": 0.30,

    "early_stopping_patience": 5,
    "checkpoint_dir_name": "checkpoints_generic-v14-antiphony",

    # ---- Section 11 ablation ----------------------------------------
    # Trains the same encoder on ONLY the Buyer's final turn. Quantifies
    # how much of the score comes from dialogue structure vs. from the
    # closing line alone.
    "run_lastturn_ablation": True,
    "ablation_encoder": "roberta-base",
    "ablation_max_len": 64,
    "ablation_epochs": 15,
    "ablation_batch_size": 16,

    # ---- Section 12 controlled-skew stress test ----------------------
    # Subsamples the TRAIN split to progressively harsher class ratios and
    # trains each twice: once with the certified imbalance machinery
    # (class-balanced focal + contrastive + tuned threshold), once with
    # plain cross-entropy. Demonstrates that the technique carried over
    # from v14 actually works on this corpus, rather than being inherited
    # configuration that never gets exercised on balanced data.
    # COST: len(skew_ratios) * 2 training runs -- the most expensive
    # section here. Set False to skip, or trim skew_ratios.
    "run_skew_stress_test": True,
    "skew_encoder": "roberta-base",
    "skew_ratios": [0.48, 0.30, 0.15, 0.10],   # fraction of 'yes' in TRAIN
    "skew_epochs": 12,                          # fewer than the main run: 8 runs
}

os.makedirs(CONFIG["out_dir"], exist_ok=True)
os.makedirs(os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"]), exist_ok=True)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "encoders"}, indent=2))
print("Encoders:", [m["hf_id"] for m in CONFIG["encoders"]])

## 2. Load the generic buying-intent dataset

`Buying_Intent_Label.csv` is generated from `Generic/final.json` with Merchant
turns emitted as `[Merchant]` and Buyer turns as `[Buyer]` — domain-accurate
names for the same two structural slots the donation notebooks marked
`[Merchant]`/`[Buyer]`. If the CSV is missing, the
cell below rebuilds it from `final.json` in place, so this notebook is
self-sufficient given only the JSON.

In [ ]:
# =============================================================
# 2. DATA DISCOVERY & LOADING
# =============================================================

def find_files_ci(roots, must_contain_all, suffix):
    """Case-insensitive recursive search -- unchanged from v3-v14."""
    must_contain_all = [t.lower() for t in must_contain_all]
    suffix = suffix.lower()
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _dirnames, filenames in os.walk(root):
            for fn in filenames:
                low = fn.lower()
                if low.endswith(suffix) and all(tok in low for tok in must_contain_all):
                    found.append(os.path.join(dirpath, fn))
    # De-duplicate by RESOLVED path: the search roots overlap ("." and
    # ".." both reach the same file), so a plain string set would report
    # one file three times and trip the single-corpus assertion below.
    seen, unique = set(), []
    for f in found:
        key = os.path.normcase(os.path.realpath(f))
        if key not in seen:
            seen.add(key)
            unique.append(f)
    return unique


# ---- Fallback: rebuild the CSV from final.json if it isn't present ----
ROLE_MARKER = {"Merchant": "[Merchant]", "Buyer": "[Buyer]"}

def build_csv_from_json(json_path, csv_path):
    """Merchant -> [Merchant] (argues), Buyer -> [Buyer] (decides).

    This is THE structural mapping the whole transfer rests on: the
    architecture's cross-attention is directional (decider queries
    arguer), so getting these two the wrong way round would invert the
    mechanism rather than merely relabel it.
    """
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    rows = []
    for i, d in enumerate(data):
        parts = []
        for t in d["turns"]:
            marker = ROLE_MARKER.get(t["role"])
            if marker is None:
                raise ValueError(f"unmapped role {t['role']!r} in {d['dialogue_id']}")
            txt = " ".join(str(t["text"]).split())
            if txt:
                parts.append(f"{marker} {txt}")
        rows.append({
            "conversation_id": str(10000 + i),
            "dialogue_id": d["dialogue_id"],
            # "\r\n " separator matches the donation corpus's own formatting,
            # so the marker regex and EDA tokenizer see an identical shape.
            "dialogue_text": "\r\n ".join(parts),
            "binary_label": str(d["labels"]["buying_intent"]).strip().lower(),
            "outcome_category": d["labels"].get("outcome_category", ""),
            "archetype": d.get("archetype", ""),
            "product": d.get("product", ""),
            "category": d.get("category", ""),
            # NOTE: source_model and turn_count are deliberately NOT emitted.
            # See CONFIG["analysis_cols"] for the rationale.
        })
    pd.DataFrame(rows).to_csv(csv_path, index=False)
    print(f"Rebuilt {csv_path} from {json_path} ({len(rows)} rows).")
    return csv_path


csv_paths = find_files_ci(CONFIG["search_roots"], CONFIG["csv_must_contain"], ".csv")

if not csv_paths:
    json_paths = find_files_ci(CONFIG["search_roots"], ["final"], ".json")
    assert json_paths, ("Neither Buying_Intent_Label.csv nor final.json found under "
                        f"{CONFIG['search_roots']}. Add the dataset as an input.")
    csv_paths = [build_csv_from_json(json_paths[0], "Buying_Intent_Label.csv")]

print(f"Found {len(csv_paths)} candidate CSV(s):")
for p in csv_paths:
    print(" -", p)

# Guard against the multi-CSV concatenation v14's looser search allowed:
# v14 searched *label*.csv and pd.concat'd every hit, so on this repo it
# would silently merge the donation corpus into this one. Paths are
# de-duplicated by realpath above, so anything left here is a genuinely
# DIFFERENT file, not the same file reached via two roots.
assert len(csv_paths) == 1, (
    f"Found {len(csv_paths)} DISTINCT buying-intent CSVs: {csv_paths}. "
    "Refusing to guess -- point CONFIG['search_roots'] at exactly one.")

raw_df = pd.read_csv(csv_paths[0])

cm = CONFIG["column_map"]
missing_cols = [c for c in cm.values() if c not in raw_df.columns]
assert not missing_cols, (
    f"Expected column(s) {missing_cols} not found. Columns present: {list(raw_df.columns)}")

df = raw_df.rename(columns={v: k for k, v in cm.items()})[list(cm.keys())].copy()

# Carry the descriptive columns along for stratification + error analysis.
for c in CONFIG["analysis_cols"]:
    if c in raw_df.columns:
        df[c] = raw_df[c].values

# De-duplicate by dialogue id (unchanged convention from v3).
orig_id_col = None
for cand in ["dialogue_id", "conversation_id", "id"]:
    if cand in raw_df.columns:
        orig_id_col = cand
        break
if orig_id_col is not None:
    df["_orig_id"] = raw_df[orig_id_col].values
    before = len(df)
    df = df.drop_duplicates(subset="_orig_id").reset_index(drop=True)
    if len(df) != before:
        print(f"Dropped {before - len(df)} duplicate row(s) by `{orig_id_col}`.")
    df = df.drop(columns="_orig_id")

df["text"] = df["text"].fillna("").astype(str)
df["binary_label"] = df["binary_label"].fillna("").astype(str).str.strip().str.lower()

bad_binary = ~df["binary_label"].isin(CONFIG["binary_classes"])
if bad_binary.any():
    print(f"WARNING: dropping {int(bad_binary.sum())} row(s) with out-of-vocabulary "
          f"binary_label: {sorted(df.loc[bad_binary, 'binary_label'].unique())}")
    df = df.loc[~bad_binary].reset_index(drop=True)

df["binary_id"] = df["binary_label"].map({c: i for i, c in enumerate(CONFIG["binary_classes"])})

# Constant zero column so the collate_fn / train loop stay LITERALLY
# identical to v14-antiphony's (which pops "modifier_labels" and discards
# it). There is no modifier task here; nothing ever reads this value.
df["modifier_id"] = 0

print(f"\nLoaded {len(df)} labeled rows after cleaning.")
df.head()

### 2b. Corpus audit — do this notebook's length budgets actually fit?

Section 9 of the donation presentation traced that project's dead modifier
head to a **data-path** bug, not an architecture one: 99.5 % of dialogues
exceeded the 256-token input budget and were truncated from the right,
discarding the very turn the label is defined by. That failure was invisible
until it was measured. This cell measures the equivalent quantities here
**before** training, so the same class of error cannot go unnoticed.

In [ ]:
# =============================================================
# 2b. CORPUS AUDIT -- verify the length budgets fit BEFORE training
# =============================================================
_MARKER_RE = re.compile(r'\[Merchant\]|\[Buyer\]')

_turns_per_dialogue, _words_per_turn, _buyer_turns, _merchant_turns = [], [], [], []
for t in df["text"]:
    ms = list(_MARKER_RE.finditer(t))
    _turns_per_dialogue.append(len(ms))
    n_dee = n_der = 0
    for i, m in enumerate(ms):
        end = ms[i + 1].start() if i + 1 < len(ms) else len(t)
        _words_per_turn.append(len(t[m.end():end].split()))
        if m.group() == "[Buyer]":
            n_dee += 1
        else:
            n_der += 1
    _buyer_turns.append(n_dee)
    _merchant_turns.append(n_der)

def _pct(arr, q):
    return float(np.percentile(np.asarray(arr), q))

audit = pd.DataFrame([
    {"quantity": "turns / dialogue",        "min": min(_turns_per_dialogue),
     "median": _pct(_turns_per_dialogue, 50), "p95": _pct(_turns_per_dialogue, 95),
     "max": max(_turns_per_dialogue), "budget": CONFIG["max_utterances"]},
    {"quantity": "words / turn",            "min": min(_words_per_turn),
     "median": _pct(_words_per_turn, 50),  "p95": _pct(_words_per_turn, 95),
     "max": max(_words_per_turn),  "budget": CONFIG["max_utt_len"]},
    {"quantity": "Buyer (decider) turns", "min": min(_buyer_turns),
     "median": _pct(_buyer_turns, 50), "p95": _pct(_buyer_turns, 95),
     "max": max(_buyer_turns), "budget": CONFIG["antiphony_max_role_len"]},
    {"quantity": "Merchant (arguer) turns", "min": min(_merchant_turns),
     "median": _pct(_merchant_turns, 50), "p95": _pct(_merchant_turns, 95),
     "max": max(_merchant_turns), "budget": CONFIG["antiphony_max_role_len"]},
])
display(audit)

_over_utt  = sum(n > CONFIG["max_utterances"] for n in _turns_per_dialogue)
_over_role = sum(n > CONFIG["antiphony_max_role_len"]
                 for n in _buyer_turns + _merchant_turns)
print(f"\nDialogues losing turns to max_utterances={CONFIG['max_utterances']}: "
      f"{_over_utt}/{len(df)} ({_over_utt/len(df):.1%})")
print(f"Role-streams truncated at max_role_len={CONFIG['antiphony_max_role_len']}: "
      f"{_over_role}/{2*len(df)}")

# The decisive check: does every dialogue END on a Buyer turn?
# The label is defined by the decider's closing position, and the GRU
# trajectory head reads its final hidden state at exactly that turn.
_ends_buyer = sum(1 for t in df["text"]
                      if list(_MARKER_RE.finditer(t))[-1].group() == "[Buyer]")
print(f"\nDialogues ending on a Buyer turn: "
      f"{_ends_buyer}/{len(df)} ({_ends_buyer/len(df):.1%})")
print("  MOST DIALOGUES END ON A MERCHANT TURN -- typically a closing courtesy\n"
      "  ('Thank you for your order, it will be dispatched today'). Two consequences,\n"
      "  both important:\n"
      "    1. A flat model that truncates from the RIGHT would read that courtesy as\n"
      "       its most recent context. The decisive Buyer move sits one turn earlier.\n"
      "       This architecture is immune by construction: the GRU trajectory head\n"
      "       reads the last turn of the PERSUADEE STREAM via pack_padded_sequence,\n"
      "       which is the last Buyer turn regardless of who speaks last overall.\n"
      "    2. The section 11 ablation therefore reads a genuinely MID-dialogue turn in\n"
      "       most cases, not a transcript-final line -- which makes it a fair control\n"
      "       rather than a trivially-advantaged one.")

### 2c. Class distribution & the majority-class floor

In [ ]:
# =============================================================
# 2c. CLASS DISTRIBUTION + MAJORITY-CLASS BASELINE
# =============================================================
print("binary_label (buying_intent) distribution:")
print(df["binary_label"].value_counts(), "\n")
print(df["binary_label"].value_counts(normalize=True).round(4), "\n")

if "outcome_category" in df.columns:
    print("outcome_category distribution (analysis only -- never a training target):")
    print(df["outcome_category"].value_counts(), "\n")
    print("outcome_category x binary_label:")
    display(pd.crosstab(df["outcome_category"], df["binary_label"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="binary_label", order=CONFIG["binary_classes"], ax=axes[0])
axes[0].set_title("buying_intent distribution")
if "outcome_category" in df.columns:
    order = df["outcome_category"].value_counts().index
    sns.countplot(data=df, y="outcome_category", order=order, ax=axes[1])
    axes[1].set_title("outcome_category distribution")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"label_distribution_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

def majority_baseline(y_true_ids, n_classes):
    majority_class = Counter(y_true_ids).most_common(1)[0][0]
    y_pred = [majority_class] * len(y_true_ids)
    return {"majority_class": majority_class,
            "accuracy": accuracy_score(y_true_ids, y_pred),
            "macro_f1": f1_score(y_true_ids, y_pred, average="macro",
                                 labels=list(range(n_classes)), zero_division=0)}

baseline_binary = majority_baseline(df["binary_id"].values, len(CONFIG["binary_classes"]))
print("\nMajority-class baseline (whole dataset; the REPORTED baseline in section 9 "
      "uses the test split only):")
print(" ", baseline_binary)
print("\nNOTE: the donation corpus was 72.6/27.4 skewed, giving a 0.841-accuracy / "
      "0.420-macro-F1\n      constant predictor. This corpus is near-balanced, so the "
      "baseline accuracy is\n      much LOWER but a model must beat it on BOTH classes "
      "to move macro-F1. Compare\n      margins over each corpus's own baseline, never "
      "the raw scores across corpora.")

## 3. Stratified 70/15/15 split

v14 stratified on `binary_label × modifier`. There is no modifier here, so
the stratum is `binary_label × outcome_category`.

Note what the §2c crosstab shows: `outcome_category` **determines**
`binary_label` exactly — `PURCHASE_COMMITTED` ⟺ `yes`, and all three of
`DEFERRED_CONSIDERATION` / `INQUIRY_DROPOUT` / `EXPLICIT_REJECTION` ⟺ `no`.
So this stratum is equivalent to stratifying on `outcome_category` alone, and
strictly finer than stratifying on the label. The benefit is real but specific:
it spreads the three *kinds* of "no" — and in particular the 11
`EXPLICIT_REJECTION` dialogues, the rarest failure mode — evenly across splits
instead of letting them land in one. That matters for §13's slice analysis,
which needs each failure mode present in the test split to say anything.

It also means `outcome_category` could not be used as a second training target
without leaking the binary label — another reason the binary-only design is
the right one here, beyond simply matching v14.

In [ ]:
# =============================================================
# 3. STRATIFIED 70/15/15 SPLIT (by binary_label x outcome_category)
# =============================================================
if "outcome_category" in df.columns:
    df["_stratum"] = df["binary_label"] + "_" + df["outcome_category"].astype(str)
else:
    df["_stratum"] = df["binary_label"]

strat_counts = df["_stratum"].value_counts()
rare = strat_counts[strat_counts < 2].index
df.loc[df["_stratum"].isin(rare), "_stratum"] = "_singleton_bucket"

train_df, temp_df = train_test_split(
    df, test_size=(CONFIG["val_frac"] + CONFIG["test_frac"]),
    stratify=df["_stratum"], random_state=SEED,
)
temp_counts = temp_df["_stratum"].value_counts()
rare2 = temp_counts[temp_counts < 2].index
temp_df = temp_df.copy()
temp_df.loc[temp_df["_stratum"].isin(rare2), "_stratum"] = "_singleton_bucket"

rel_test_size = CONFIG["test_frac"] / (CONFIG["val_frac"] + CONFIG["test_frac"])
val_df, test_df = train_test_split(
    temp_df, test_size=rel_test_size,
    stratify=temp_df["_stratum"], random_state=SEED,
)

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: n={len(split)} ({len(split)/len(df):.1%})")

print("\nbinary_label composition per split:")
display(pd.concat({
    "train": train_df["binary_label"].value_counts(normalize=True),
    "val":   val_df["binary_label"].value_counts(normalize=True),
    "test":  test_df["binary_label"].value_counts(normalize=True),
}, axis=1).round(4))

if "outcome_category" in df.columns:
    print("\noutcome_category composition per split:")
    display(pd.concat({
        "train": train_df["outcome_category"].value_counts(),
        "val":   val_df["outcome_category"].value_counts(),
        "test":  test_df["outcome_category"].value_counts(),
    }, axis=1).fillna(0).astype(int))

for split in (train_df, val_df, test_df):
    split.drop(columns=["_stratum"], inplace=True)
df.drop(columns=["_stratum"], inplace=True)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

## 3b. EDA augmentation — machinery present, **disabled by default**

v14's augmentation plan is class-conditional: `conditional` ×6, `deferred` ×2,
`no` ×1, sized to repair a corpus where one class held 2.1 % of the rows.
This corpus is **52.4 / 47.6** — running that plan would oversample `no` and
*manufacture* the imbalance it exists to remove.

The marker-safe machinery is kept below, with two adaptations, so enabling it
is a one-flag change (`CONFIG["use_augmentation"] = True`):

- **Uniform copies across both classes** (`aug_num_per_row`), not per-class — the
  balance is preserved rather than perturbed.
- **Commerce vocabulary.** v14's `PROTECTED_WORDS` locked donation terms
  (`donate`, `charity`, `pledge`) and its `_SYNONYMS` mapped donation phrasing.
  On this corpus those lists are inert, so the protected set now locks the
  words that *carry* buying intent (`order`, `confirm`, `later`, `think`,
  negations, prices) and the synonym table covers commerce phrasing.

In [ ]:
# =============================================================
# 3b. EDA AUGMENTATION  (marker-safe; commerce vocabulary; OFF by default)
# =============================================================
import random as _random
import hashlib

# Locked words: never replaced, deleted, or swapped. These are the tokens
# that DEFINE the buying-intent label -- perturbing them would relabel the
# dialogue, which is exactly the bug v5 fixed on the donation corpus.
PROTECTED_WORDS = {
    # deferral / conditionality -- the DEFERRED_CONSIDERATION signal
    "if", "unless", "provided", "later", "think", "thinking", "when", "then",
    "tomorrow", "soon", "after", "once", "maybe", "may", "might", "could",
    "would", "will", "discuss", "check", "decide", "consider",
    # commitment -- the PURCHASE_COMMITTED signal
    "order", "confirm", "confirmed", "buy", "buying", "purchase", "take",
    "deal", "done", "placing", "place", "book", "booking", "cod", "advance",
    # rejection -- the EXPLICIT_REJECTION signal
    "not", "no", "never", "don't", "dont", "won't", "wont", "can't", "cant",
    "cancel", "expensive", "budget",
    # transactional nouns whose replacement would distort meaning
    "price", "taka", "tk", "warranty", "delivery", "stock", "discount",
}
SPEAKER_MARKERS = {"[merchant]", "[buyer]"}

_SYNONYMS = {
    "good": ["nice", "great", "fine", "decent"],
    "really": ["truly", "genuinely", "honestly"],
    "want": ["would like", "wish", "need"],
    "help": ["assist", "support", "aid"],
    "sure": ["certain", "confident", "positive"],
    "sorry": ["apologies", "regret", "my bad"],
    "okay": ["alright", "fine", "ok"],
    "thanks": ["thank you", "appreciate it", "cheers"],
    "great": ["awesome", "wonderful", "excellent"],
    "problem": ["issue", "trouble", "difficulty"],
    "product": ["item", "unit", "model"],
    "available": ["in stock", "on hand", "ready"],
    "send": ["ship", "dispatch", "forward"],
    "fast": ["quick", "rapid", "speedy"],
    "genuine": ["authentic", "original", "legit"],
    "quality": ["build", "standard", "grade"],
    "shop": ["store", "outlet", "showroom"],
    "customer": ["client", "buyer", "patron"],
    "offer": ["deal", "promotion", "package"],
    "brand": ["make", "label", "manufacturer"],
}

def _get_synonym(word):
    return _random.choice(_SYNONYMS[word.lower()]) if word.lower() in _SYNONYMS else None

# Markers matched as atomic tokens FIRST, regardless of what they are glued
# to -- v5's fix. text.split(" ") corrupted 15.9% of augmented rows on the
# donation corpus because 57.5% of markers are glued to a preceding "\r\n".
_TOKEN_PATTERN = re.compile(r'\[Merchant\]|\[Buyer\]|\S+')

def _tokenize_protecting_markers(text):
    return _TOKEN_PATTERN.findall(text)

def _is_locked(tok):
    low = tok.strip(".,!?;:\"'").lower()
    # Numbers are locked too: prices and quantities carry transactional meaning.
    if any(ch.isdigit() for ch in tok):
        return True
    return low in PROTECTED_WORDS or low in SPEAKER_MARKERS or tok.lower() in SPEAKER_MARKERS

def _tag_token_roles(words):
    """Tag each token with its enclosing speaker turn -- used ONLY to bias
    which words augmentation prefers to touch. Has no effect on the model's
    role handling, which is assigned per TURN in split_into_utterances()."""
    roles, current = [], "other"
    for w in words:
        low = w.lower()
        if low == "[merchant]":
            current = "merchant"
        elif low == "[buyer]":
            current = "buyer"
        roles.append(current)
    return roles

def _ranked_candidates(indices, roles, bias_weight):
    buyer = [i for i in indices if roles[i] == "buyer"]
    other     = [i for i in indices if roles[i] != "buyer"]
    _random.shuffle(buyer); _random.shuffle(other)
    return buyer + other if _random.random() < bias_weight else other + buyer

def eda_synonym_replacement(words, n, roles=None, bias_weight=0.0):
    new_words = words.copy()
    candidates = [i for i, w in enumerate(new_words) if not _is_locked(w) and _get_synonym(w)]
    if roles is not None and bias_weight > 0:
        candidates = _ranked_candidates(candidates, roles, bias_weight)
    else:
        _random.shuffle(candidates)
    replaced = 0
    for idx in candidates:
        syn = _get_synonym(new_words[idx])
        if syn:
            new_words[idx] = syn
            replaced += 1
        if replaced >= n:
            break
    return new_words

def eda_random_deletion(words, p):
    if len(words) <= 3:
        return words.copy()
    new_words = [w for w in words if _is_locked(w) or _random.random() > p]
    return new_words if new_words else [words[_random.randrange(len(words))]]

def eda_random_swap(words, n):
    new_words = words.copy()
    swappable = [i for i, w in enumerate(new_words) if not _is_locked(w)]
    for _ in range(n):
        if len(swappable) < 2:
            break
        i, j = _random.sample(swappable, 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return new_words

def eda_random_insertion(words, n, roles=None, bias_weight=0.0):
    new_words = words.copy()
    base_candidates = [i for i, w in enumerate(words) if not _is_locked(w) and _get_synonym(w)]
    for _ in range(n):
        if not base_candidates:
            break
        idx = (_ranked_candidates(base_candidates, roles, bias_weight)[0]
               if roles is not None and bias_weight > 0 else _random.choice(base_candidates))
        new_words.insert(_random.randrange(len(new_words) + 1), _get_synonym(words[idx]))
    return new_words

def eda_augment_one(text, alpha=0.15, seed=None, bias_weight=0.0):
    if seed is not None:
        _random.seed(seed)
    words = _tokenize_protecting_markers(text)
    roles = _tag_token_roles(words) if bias_weight > 0 else None
    n_ops = max(1, int(alpha * len(words)))
    op = _random.choice(["synonym", "swap", "delete", "insert"])
    if op == "synonym":
        words = eda_synonym_replacement(words, n_ops, roles=roles, bias_weight=bias_weight)
    elif op == "swap":
        words = eda_random_swap(words, n_ops)
    elif op == "delete":
        words = eda_random_deletion(words, p=alpha)
    else:
        words = eda_random_insertion(words, n_ops, roles=roles, bias_weight=bias_weight)
    return " ".join(words)

# sha256, not Python's hash(): built-in string hashing is randomized
# per-process (PYTHONHASHSEED), which silently produced different augmented
# text on every kernel restart despite a fixed SEED.
def _stable_seed(*parts):
    return int(hashlib.sha256("::".join(str(p) for p in parts).encode("utf-8")).hexdigest()[:8], 16)

def augment_dataframe_balanced(df_in, config, seed=42):
    """CHANGED from v14: uniform copies for EVERY row regardless of class,
    so a balanced corpus stays balanced. v14's per-class plan existed to
    repair a 2.1% minority that has no counterpart here."""
    _random.seed(seed)
    n_copies = config.get("aug_num_per_row", 1)
    bias_weight = config.get("aug_buyer_bias_weight", 0.0)
    rows_to_add = []
    for _, row in df_in.iterrows():
        for k in range(n_copies):
            new_row = row.copy()
            new_row["text"] = eda_augment_one(
                row["text"], alpha=config["aug_alpha"],
                seed=_stable_seed(row["text"], k), bias_weight=bias_weight)
            rows_to_add.append(new_row)
    if not rows_to_add:
        return df_in.reset_index(drop=True)
    out = pd.concat([df_in, pd.DataFrame(rows_to_add)], ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


if CONFIG.get("use_augmentation", False):
    _before_n = len(train_df)
    train_df = augment_dataframe_balanced(train_df, CONFIG, seed=SEED)
    print(f"Train split: {_before_n} -> {len(train_df)} rows after balanced EDA augmentation")
    print(train_df["binary_label"].value_counts())

    # Marker-integrity self-check (v5's QA gate) -- must be 0 mismatches.
    _n_bad = sum(1 for _, r in train_df.iterrows()
                 if len(_MARKER_RE.findall(r["text"])) == 0)
    print(f"Rows with zero speaker markers after augmentation: {_n_bad} (must be 0)")
else:
    print("Augmentation DISABLED (CONFIG['use_augmentation'] = False).")
    print("Rationale: this corpus is near-balanced (52.4/47.6), so v14's "
          "class-conditional\noversampling plan would introduce imbalance rather "
          "than repair it. The machinery\nabove is commerce-adapted and ready; "
          "flip the flag for a follow-up run.")
    print("\nTrain split binary_label distribution (unaugmented):")
    print(train_df["binary_label"].value_counts())

## 4. Dataset, utterance splitting & hierarchical collate

**Structurally identical to v14-antiphony.** Only the marker literals and the
two role-constant names changed: `[Merchant]`/`[Buyer]` and
`ROLE_MERCHANT`=0 / `ROLE_BUYER`=1 occupy exactly the slots
`[Persuader]`/`[Persuadee]` and `ROLE_PERSUADER`/`ROLE_PERSUADEE` did. Every
tensor shape, mask convention and code path is unchanged, so the model class
below still binds to them using v14's own internal variable names.

In [ ]:
# =============================================================
# 4. TORCH DATASET + UTTERANCE SPLITTING + HIERARCHICAL COLLATE
#    (verbatim from v14-antiphony -- the marker convention makes it
#     domain-agnostic)
# =============================================================
_SPEAKER_PATTERN = re.compile(r'\[Merchant\]|\[Buyer\]')
ROLE_MERCHANT = 0    # Merchant here -- the party that ARGUES
ROLE_BUYER = 1    # Buyer here    -- the party that DECIDES
ROLE_PAD       = 2    # padding turn-slot


class DialogueIntentDataset(Dataset):
    def __init__(self, dataframe):
        self.texts = dataframe["text"].tolist()
        self.binary_ids = dataframe["binary_id"].tolist()
        self.modifier_ids = dataframe["modifier_id"].tolist()   # constant 0, unused

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return {"text": self.texts[idx],
                "binary_id": self.binary_ids[idx],
                "modifier_id": self.modifier_ids[idx]}


def split_into_utterances(text, max_utterances=32):
    """Split a dialogue into (role, utterance_text) turns by regex POSITION,
    so it is robust to markers glued to surrounding whitespace/punctuation.

    If a dialogue exceeds max_utterances, the EARLIEST turns are dropped and
    the most recent are kept -- intent is resolved near the end of a
    conversation. On this corpus max turns = 32, so this branch never fires
    (verified in the section 2b audit)."""
    matches = list(_SPEAKER_PATTERN.finditer(text))
    utterances = []
    for i, m in enumerate(matches):
        role = ROLE_MERCHANT if m.group() == "[Merchant]" else ROLE_BUYER
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        utt_text = text[start:end].strip()
        if utt_text:
            utterances.append((role, utt_text))
    if len(utterances) > max_utterances:
        utterances = utterances[-max_utterances:]
    return utterances


def make_hier_collate_fn(tokenizer, max_utt_len, max_utterances):
    """Builds a batch of shape [B, U, L] plus utt_mask [B, U] and
    role_ids [B, U]. Padding turn-slots get a single valid dummy token
    (CLS/BOS, attention_mask=1 at position 0) rather than an all-zero mask,
    so no encoder implementation has to handle a fully-masked sequence;
    utt_mask excludes those slots downstream regardless."""
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    cls_id = tokenizer.cls_token_id
    if cls_id is None:
        cls_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else pad_id

    def collate_fn(batch):
        all_utts = []
        for b in batch:
            utts = split_into_utterances(b["text"], max_utterances=max_utterances)
            if not utts:
                utts = [(ROLE_BUYER, "")]   # degenerate fallback
            all_utts.append(utts)

        B = len(all_utts)
        U = max(len(u) for u in all_utts)

        flat_texts, flat_roles, owner = [], [], []
        for di, utts in enumerate(all_utts):
            for ui, (role, text) in enumerate(utts):
                flat_texts.append(text if text.strip() else " ")
                flat_roles.append(role)
                owner.append((di, ui))

        enc = tokenizer(flat_texts, padding=True, truncation=True,
                        max_length=max_utt_len, return_tensors="pt")
        L = enc["input_ids"].shape[1]

        input_ids = torch.full((B, U, L), pad_id, dtype=torch.long)
        attn_mask = torch.zeros(B, U, L, dtype=torch.long)
        utt_mask  = torch.zeros(B, U, dtype=torch.long)
        role_ids  = torch.full((B, U), ROLE_PAD, dtype=torch.long)

        input_ids[:, :, 0] = cls_id

        for flat_idx, (di, ui) in enumerate(owner):
            input_ids[di, ui] = enc["input_ids"][flat_idx]
            attn_mask[di, ui] = enc["attention_mask"][flat_idx]
            utt_mask[di, ui]  = 1
            role_ids[di, ui]  = flat_roles[flat_idx]

        pad_slot = (utt_mask == 0)
        attn_mask[:, :, 0] = attn_mask[:, :, 0].masked_fill(pad_slot, 1)

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "utt_mask": utt_mask,
            "role_ids": role_ids,
            "binary_labels":   torch.tensor([b["binary_id"]   for b in batch], dtype=torch.long),
            "modifier_labels": torch.tensor([b["modifier_id"] for b in batch], dtype=torch.long),
        }
    return collate_fn


# ---- Sanity check: the collate output on a real batch -----------------
_probe_tok = AutoTokenizer.from_pretrained("roberta-base")
_probe_collate = make_hier_collate_fn(_probe_tok, CONFIG["max_utt_len"], CONFIG["max_utterances"])
_probe_ds = DialogueIntentDataset(train_df.head(4))
_probe_batch = _probe_collate([_probe_ds[i] for i in range(4)])
print("collate output shapes:")
for k, v in _probe_batch.items():
    print(f"  {k:18s} {tuple(v.shape)}")
print("\nrole_ids[0] (0=Merchant/arguer, 1=Buyer/decider, 2=pad):")
print("  ", _probe_batch["role_ids"][0].tolist())
print("utt_mask[0]:")
print("  ", _probe_batch["utt_mask"][0].tolist())
del _probe_tok, _probe_collate, _probe_ds, _probe_batch

## 5. Model — dual-stream arguer/decider cross-attention (v14-antiphony, **unchanged**)

```
input_ids, attention_mask   [B, U, L]
        |
   flatten [B*U, L], ONE batched pass through the shared utterance
   encoder (RoBERTa-base | DeBERTa-v3-base | TOD-BERT)
        |
   masked mean-pool per turn -> [B, U, H]
        |
   split by role  ----------------------------+
        |                                     |
   Merchant (arguer) stream [B,R,H]      Buyer (decider) stream [B,R,H]
   + within-stream position emb          + within-stream position emb
   2-layer Transformer                   2-layer Transformer
        |                                     |
        +---- key/value ---> cross-attention <--- query
                                              |
                          residual + LayerNorm  [B, R, H]
                                              |
                      +-----------------------+-----------------------+
                      |                                               |
            attention pooling                                trajectory GRU
          "what mattered most"                          final hidden state at the
                 [B, H]                                  true last Buyer turn [B, H]
                      |                                               |
                      +------------------ concat --------------------+
                                          [B, 2H]
                                             |
                                        binary head -> logits [B, 2]
```

The three mechanisms, and what each means **in this domain**:

1. **Speaker-disentangled streams.** The Merchant's pitch develops across
   Merchant turns; the Buyer's stance develops across Buyer turns. Each gets
   its own Transformer over its own chronological turns, uninterleaved.
2. **Directional cross-attention.** Buyer queries Merchant — "which of the
   merchant's claims is this buyer's position responding to". Asymmetric by
   construction; reversing the direction would be a different model.
3. **Commitment-trajectory GRU.** Read at each dialogue's true final Buyer
   turn via `pack_padded_sequence`, never a padding slot. A GRU's hidden
   state is inherently recency-weighted, so "did this buyer converge toward
   committing or toward deferring" emerges from the mechanism rather than
   from a hand-tuned recency scalar.

**Not one line of this class differs from `capstone-v14-antiphony.ipynb`.**
That is deliberate: if the architecture were edited for this corpus, a
transfer result would no longer be attributable to the architecture.

In [ ]:
# =============================================================
# 5. DUAL-STREAM ARGUER/DECIDER CROSS-ATTENTION
#    (verbatim from v14-antiphony -- NOT modified for this domain)
# =============================================================

class AntiphonyClassifier(nn.Module):
    """
    Dual-stream, speaker-disentangled encoder for two-party dialogues in
    which one party argues and the other decides.

    After the shared per-turn utterance encoding, turn vectors are split by
    role into an arguer stream and a decider stream, each in its own
    chronological order, each given its OWN small Transformer -- modelling
    how each party's own position develops independent of interleaving
    (same family of idea as DialogueRNN's per-party state tracking,
    Majumder et al. 2019).

    The decider stream then cross-attends over the arguer stream (decider as
    query, arguer as key/value) -- an explicit, directional "how does the
    decision-maker's stance respond to what was argued" mechanism
    ("antiphony": alternating call-and-response between two distinct voices).

    A GRU over the cross-attended decider stream gives a "commitment
    trajectory" from its FINAL hidden state, concatenated with an
    attention-pooled summary of the same stream before the binary head.
    Returns (binary_logits, shared).
    """

    ROLE_ARGUER_ID = 0   # arguer   (Merchant in this corpus)
    ROLE_DECIDER_ID = 1   # decider  (Buyer in this corpus)

    def __init__(self, hf_id, n_binary, dropout=0.1,
                 stream_layers=2, stream_heads=8, cross_heads=8, max_role_len=20):
        super().__init__()
        self.utt_encoder = AutoModel.from_pretrained(hf_id)
        if hasattr(self.utt_encoder, "gradient_checkpointing_enable"):
            self.utt_encoder.gradient_checkpointing_enable()
        hidden = self.utt_encoder.config.hidden_size
        self.max_role_len = max_role_len
        self.dropout = nn.Dropout(dropout)

        # Within-stream chronological position. Each stream is role-pure,
        # so a role embedding would be redundant; what matters is "this is
        # this speaker's Nth turn".
        self.stream_pos_embedding = nn.Embedding(max_role_len, hidden)
        nn.init.normal_(self.stream_pos_embedding.weight, std=0.02)

        persuader_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=stream_heads, dim_feedforward=hidden * 4,
            dropout=dropout, activation="gelu", batch_first=True)
        self.persuader_transformer = nn.TransformerEncoder(persuader_layer, num_layers=stream_layers)

        persuadee_layer = nn.TransformerEncoderLayer(
            d_model=hidden, nhead=stream_heads, dim_feedforward=hidden * 4,
            dropout=dropout, activation="gelu", batch_first=True)
        self.persuadee_transformer = nn.TransformerEncoder(persuadee_layer, num_layers=stream_layers)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden, num_heads=cross_heads, dropout=dropout, batch_first=True)
        self.cross_norm = nn.LayerNorm(hidden)

        self.attn_proj  = nn.Linear(hidden, hidden)
        self.attn_score = nn.Linear(hidden, 1, bias=False)

        self.trajectory_gru = nn.GRU(hidden, hidden, batch_first=True)

        self.binary_head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_binary),
        )

    def _encode_turns(self, input_ids, attention_mask):
        """[B, U, L] -> [B, U, H]: one flattened forward pass through the
        shared utterance encoder + masked mean-pool per turn."""
        B, U, L = input_ids.shape
        flat_ids  = input_ids.view(B * U, L)
        flat_mask = attention_mask.view(B * U, L)

        outputs = self.utt_encoder(input_ids=flat_ids, attention_mask=flat_mask)
        flat_hidden = outputs.last_hidden_state             # [B*U, L, H]

        mask = flat_mask.unsqueeze(-1).float()
        summed = (flat_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        turn_vecs = summed / counts                         # [B*U, H]
        return turn_vecs.view(B, U, -1)

    def _split_by_role(self, turn_vecs, role_ids, utt_mask, role_id_value):
        """Returns (role_vecs [B, R, H], role_mask [B, R]) for one role,
        right-padded, keeping the LAST max_role_len real turns of that role
        in original chronological order."""
        B, U, H = turn_vecs.shape
        R = self.max_role_len
        device = turn_vecs.device
        out = turn_vecs.new_zeros(B, R, H)
        out_mask = torch.zeros(B, R, dtype=torch.bool, device=device)
        role_match = (role_ids == role_id_value) & (utt_mask == 1)
        for b in range(B):
            idx = torch.nonzero(role_match[b], as_tuple=True)[0]
            if idx.numel() > R:
                idx = idx[-R:]
            n = idx.numel()
            if n > 0:
                out[b, :n] = turn_vecs[b, idx]
                out_mask[b, :n] = True
        return out, out_mask

    def forward(self, input_ids, attention_mask, utt_mask, role_ids, **_):
        turn_vecs = self._encode_turns(input_ids, attention_mask)   # [B, U, H]

        persuader_vecs, persuader_mask = self._split_by_role(
            turn_vecs, role_ids, utt_mask, self.ROLE_ARGUER_ID)
        persuadee_vecs, persuadee_mask = self._split_by_role(
            turn_vecs, role_ids, utt_mask, self.ROLE_DECIDER_ID)

        R = self.max_role_len
        stream_positions = torch.arange(R, device=turn_vecs.device).unsqueeze(0)
        pos_emb = self.stream_pos_embedding(stream_positions)       # [1, R, H]
        persuader_vecs = persuader_vecs + pos_emb
        persuadee_vecs = persuadee_vecs + pos_emb

        persuader_encoded = self.persuader_transformer(
            persuader_vecs, src_key_padding_mask=~persuader_mask)   # [B, R, H]
        persuadee_encoded = self.persuadee_transformer(
            persuadee_vecs, src_key_padding_mask=~persuadee_mask)   # [B, R, H]

        # Decider queries arguer: "what did the merchant claim that's
        # relevant to how this buyer responds".
        cross_out, _ = self.cross_attn(
            query=persuadee_encoded, key=persuader_encoded, value=persuader_encoded,
            key_padding_mask=~persuader_mask)
        persuadee_final = self.cross_norm(persuadee_encoded + cross_out)   # [B, R, H]

        # (a) Attention-pooled summary -- "what mattered most, overall".
        scores = self.attn_score(torch.tanh(self.attn_proj(persuadee_final))).squeeze(-1)
        scores = scores.masked_fill(~persuadee_mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        pooled_summary = (persuadee_final * weights).sum(dim=1)     # [B, H]

        # (b) Commitment trajectory: GRU final hidden state at each row's
        # true last valid decider turn -- never a padding slot.
        lengths = persuadee_mask.sum(dim=1).clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            persuadee_final, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.trajectory_gru(packed)
        trajectory_vec = h_n.squeeze(0)                             # [B, H]

        shared = self.dropout(torch.cat([pooled_summary, trajectory_vec], dim=-1))  # [B, 2H]
        binary_logits = self.binary_head(shared)
        return binary_logits, shared


print("AntiphonyClassifier defined (architecture identical to v14-antiphony).")

## 6. Loss — class-balanced focal + supervised contrastive (**unchanged**)

Kept verbatim from v14 even though this corpus is near-balanced. The
class-balanced weights will come out close to `[1.0, 1.0]` here — which is
the correct behaviour, not a no-op worth removing: keeping the loss identical
means any difference from the donation run is attributable to the data, not
to a changed objective. The printed weights below make the near-uniformity
explicit.

In [ ]:
# =============================================================
# 6. CLASS-BALANCED FOCAL LOSS + SUPERVISED CONTRASTIVE LOSS (unchanged)
# =============================================================

class ClassBalancedFocalLoss(nn.Module):
    """Cui et al. (2019) class-balanced re-weighting (effective number of
    samples) combined with the focal focusing term (Lin et al., 2017)."""
    def __init__(self, samples_per_class, beta=0.999, gamma=2.0):
        super().__init__()
        samples_per_class = np.asarray(samples_per_class, dtype=np.float64)
        samples_per_class = np.clip(samples_per_class, 1.0, None)
        effective_num = 1.0 - np.power(beta, samples_per_class)
        effective_num = np.where(effective_num <= 0, 1e-8, effective_num)
        weights = (1.0 - beta) / effective_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.register_buffer("class_weights", torch.tensor(weights, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, targets):
        if logits.size(0) == 0:
            return torch.zeros((), device=logits.device)
        log_probs = F.log_softmax(logits, dim=-1)
        probs = log_probs.exp()
        pt     = probs.gather(1, targets.unsqueeze(1)).squeeze(1).clamp(min=1e-8)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        alpha_t = self.class_weights.to(logits.device)[targets]
        return (-alpha_t * (1 - pt).pow(self.gamma) * log_pt).mean()


def supervised_contrastive_loss(pooled, labels, temperature=0.1):
    """In-batch supervised contrastive loss (Khosla et al., 2020), single-view.
    Anchors with no same-class partner in the batch contribute nothing."""
    n = pooled.size(0)
    if n < 2:
        return torch.zeros((), device=pooled.device)

    z = F.normalize(pooled, dim=-1)
    sim = torch.matmul(z, z.T) / temperature

    labels = labels.view(-1, 1)
    same_class = (labels == labels.T).float()
    self_mask  = torch.eye(n, device=z.device)
    positive_mask = same_class - self_mask

    logits_max, _ = sim.max(dim=1, keepdim=True)
    logits = sim - logits_max.detach()
    exp_logits = torch.exp(logits) * (1 - self_mask)
    log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

    pos_counts = positive_mask.sum(dim=1)
    valid = pos_counts > 0
    if valid.sum() == 0:
        return torch.zeros((), device=pooled.device)
    return -((positive_mask * log_prob).sum(dim=1)[valid] / pos_counts[valid]).mean()


def samples_per_class(ids, n_classes):
    counts = np.bincount(ids, minlength=n_classes).astype(float)
    counts[counts == 0] = 1.0
    return counts

binary_samples_per_class = samples_per_class(train_df["binary_id"].values,
                                             len(CONFIG["binary_classes"]))
print("buying_intent samples/class (train):",
      dict(zip(CONFIG["binary_classes"], binary_samples_per_class.tolist())))

binary_focal_loss_fn = ClassBalancedFocalLoss(
    binary_samples_per_class, beta=CONFIG["cb_beta"], gamma=CONFIG["focal_gamma"])

print(f"\nClass-balanced weights (beta={CONFIG['cb_beta']}):")
print(" ", dict(zip(CONFIG["binary_classes"],
                    [round(w, 4) for w in binary_focal_loss_fn.class_weights.tolist()])))
print("  (near-uniform, as expected on a balanced corpus -- on the donation corpus "
      "these\n   were strongly skewed. The loss is deliberately left unchanged so the "
      "objective is\n   not a confound in the transfer comparison.)")

## 7. Training / evaluation functions (**unchanged from v14-antiphony**)

Checkpoint selection on validation binary macro-F1, early stopping at
patience 5, and a val-tuned decision-threshold search reported alongside the
untuned 0.5 result. Reporting both matters here: with only 75 validation
rows, a tuned threshold can overfit the validation set, so the 0.5 column is
the conservative number.

In [ ]:
# =============================================================
# 7. TRAIN / EVAL FUNCTIONS  (binary-only, unchanged from v14-antiphony)
# =============================================================

def compute_losses(binary_logits, shared, binary_labels, include_contrastive):
    binary_loss = binary_focal_loss_fn(binary_logits, binary_labels)
    total = CONFIG["binary_loss_weight"] * binary_loss
    if include_contrastive and CONFIG.get("use_contrastive", False):
        con_binary = supervised_contrastive_loss(
            shared, binary_labels, temperature=CONFIG["contrastive_temperature"])
        total = total + CONFIG["contrastive_weight_binary"] * con_binary
    return total


def evaluate(model, loader, device, binary_threshold=0.5):
    """P(yes) >= binary_threshold -> predict 'yes'."""
    model.eval()
    all_binary_true, all_binary_pred, all_binary_probs = [], [], []
    total_loss, n_batches = 0.0, 0
    yes_idx = CONFIG["binary_classes"].index("yes")

    with torch.no_grad():
        for batch in loader:
            binary_labels = batch.pop("binary_labels").to(device)
            batch.pop("modifier_labels")           # constant 0, unused
            batch = {k: v.to(device) for k, v in batch.items()}

            binary_logits, shared = model(**batch)

            # Eval loss excludes the contrastive term: it is a
            # representation-shaping regulariser, not a prediction-quality number.
            loss = compute_losses(binary_logits, shared, binary_labels,
                                  include_contrastive=False)
            total_loss += loss.item()
            n_batches += 1

            binary_probs_yes = torch.softmax(binary_logits, dim=-1)[:, yes_idx]
            binary_preds = (binary_probs_yes >= binary_threshold).long()

            all_binary_true.extend(binary_labels.cpu().tolist())
            all_binary_pred.extend(binary_preds.cpu().tolist())
            all_binary_probs.extend(binary_probs_yes.cpu().tolist())

    return {
        "loss": total_loss / max(n_batches, 1),
        "binary_accuracy": accuracy_score(all_binary_true, all_binary_pred),
        "binary_f1": f1_score(all_binary_true, all_binary_pred, average="binary",
                              pos_label=yes_idx, zero_division=0),
        "binary_macro_f1": f1_score(all_binary_true, all_binary_pred, average="macro",
                                    labels=[0, 1], zero_division=0),
        "binary_true": all_binary_true,
        "binary_pred": all_binary_pred,
        "binary_probs": all_binary_probs,
    }


def find_optimal_binary_threshold(model, val_loader, device):
    """Grid-search the threshold maximising macro-F1 on validation."""
    model.eval()
    all_true, all_probs = [], []
    yes_idx = CONFIG["binary_classes"].index("yes")
    with torch.no_grad():
        for batch in val_loader:
            binary_labels = batch.pop("binary_labels").to(device)
            batch.pop("modifier_labels")
            batch = {k: v.to(device) for k, v in batch.items()}
            binary_logits, _ = model(**batch)
            probs = torch.softmax(binary_logits, dim=-1)[:, yes_idx]
            all_true.extend(binary_labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    best_threshold, best_macro_f1 = 0.5, 0.0
    for t in [v / 100 for v in range(10, 91, 5)]:
        preds = [1 if p >= t else 0 for p in all_probs]
        m = f1_score(all_true, preds, average="macro", labels=[0, 1], zero_division=0)
        if m > best_macro_f1:
            best_macro_f1, best_threshold = m, t
    return best_threshold, best_macro_f1


def train_one_encoder(encoder_cfg, train_df, val_df, test_df):
    name, hf_id = encoder_cfg["name"], encoder_cfg["hf_id"]
    sep_line = "=" * 70
    print(f"\n{sep_line}\nTraining {name} ({hf_id})  [antiphony, buying-intent]"
          f"  [max_role_len={CONFIG['antiphony_max_role_len']}, "
          f"stream_layers={CONFIG['antiphony_stream_layers']}]"
          f"  [contrastive={'ON' if CONFIG.get('use_contrastive') else 'OFF'}]\n{sep_line}")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    collate_fn = make_hier_collate_fn(tokenizer, CONFIG["max_utt_len"],
                                      CONFIG["max_utterances"])

    train_ds = DialogueIntentDataset(train_df)
    val_ds   = DialogueIntentDataset(val_df)
    test_ds  = DialogueIntentDataset(test_df)

    enc_bs      = encoder_cfg.get("batch_size", CONFIG["batch_size"])
    enc_eval_bs = encoder_cfg.get("eval_batch_size", CONFIG["eval_batch_size"])
    train_loader = DataLoader(train_ds, batch_size=enc_bs, shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(val_ds,  batch_size=enc_eval_bs, shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(test_ds, batch_size=enc_eval_bs, shuffle=False, collate_fn=collate_fn)

    model = AntiphonyClassifier(
        hf_id,
        n_binary=len(CONFIG["binary_classes"]),
        stream_layers=CONFIG["antiphony_stream_layers"],
        stream_heads=CONFIG["antiphony_stream_heads"],
        cross_heads=CONFIG["antiphony_cross_heads"],
        max_role_len=CONFIG["antiphony_max_role_len"],
    ).to(DEVICE)

    encoder_lr = encoder_cfg.get("lr", CONFIG["lr"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=encoder_lr,
                                  weight_decay=CONFIG["weight_decay"], eps=1e-6)
    total_steps = len(train_loader) * CONFIG["num_epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * encoder_cfg.get("warmup_ratio", CONFIG["warmup_ratio"])),
        num_training_steps=total_steps,
    )

    ckpt_path = os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"], f"{name}_best.pt")
    best_val_score, epochs_without_improve, history = -1.0, 0, []

    for epoch in range(1, CONFIG["num_epochs"] + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"[{name}] epoch {epoch}/{CONFIG['num_epochs']}")

        for batch in pbar:
            binary_labels = batch.pop("binary_labels").to(DEVICE)
            batch.pop("modifier_labels")
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()
            binary_logits, shared = model(**batch)
            loss = compute_losses(binary_logits, shared, binary_labels,
                                  include_contrastive=True)

            if torch.isnan(loss):
                print("  [WARNING] NaN loss detected -- skipping batch.")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        val_metrics = evaluate(model, val_loader, DEVICE, binary_threshold=0.5)
        val_score = val_metrics["binary_macro_f1"]
        history.append({
            "epoch": epoch,
            "train_loss": running_loss / len(train_loader),
            "val_loss": val_metrics["loss"],
            "val_binary_f1": val_metrics["binary_f1"],
            "val_binary_macro_f1": val_metrics["binary_macro_f1"],
            "val_binary_accuracy": val_metrics["binary_accuracy"],
        })
        print(f"  epoch {epoch}: train_loss={history[-1]['train_loss']:.4f} "
              f"val_binary_macroF1={val_metrics['binary_macro_f1']:.4f}")

        if val_score > best_val_score:
            best_val_score, epochs_without_improve = val_score, 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  -> new best (binary_macroF1={val_score:.4f}), checkpoint saved.")
        else:
            epochs_without_improve += 1
            if epochs_without_improve >= CONFIG["early_stopping_patience"]:
                print(f"  -> no improvement for {epochs_without_improve} epochs, stopping early.")
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))

    best_thresh, thresh_val_macro_f1 = find_optimal_binary_threshold(model, val_loader, DEVICE)
    print(f"\n  Threshold search -> best thresh={best_thresh:.2f} "
          f"(val macro-F1={thresh_val_macro_f1:.4f})")

    test_m_default = evaluate(model, test_loader, DEVICE, binary_threshold=0.5)
    test_m_tuned   = evaluate(model, test_loader, DEVICE, binary_threshold=best_thresh)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name": name, "hf_id": hf_id, "history": history,
        "test_metrics": test_m_tuned,
        "test_metrics_default": test_m_default,
        "test_binary_accuracy": test_m_tuned["binary_accuracy"],
        "test_binary_f1": test_m_tuned["binary_f1"],
        "test_binary_macro_f1": test_m_tuned["binary_macro_f1"],
        "best_threshold": best_thresh,
        "ckpt_path": ckpt_path,
    }

## 8. Run training for all three encoders

In [ ]:
# =============================================================
# 8. RUN ALL THREE ENCODERS
# =============================================================
results = {}
for encoder_cfg in CONFIG["encoders"]:
    results[encoder_cfg["name"]] = train_one_encoder(encoder_cfg, train_df, val_df, test_df)

print("\nDone training all encoders:", list(results.keys()))

In [ ]:
# =============================================================
# 8b. THRESHOLD COMPARISON TABLE
# =============================================================
print(f"\n{'='*78}")
print("Binary head: default threshold (0.5) vs val-tuned threshold")
print(f"{'='*78}")
print(f"{'Model':<26} {'Thresh':>6}  {'Accuracy':>9}  {'MacroF1':>8}  {'no F1':>8}  {'yes F1':>8}")
print("-" * 78)
for name, r in results.items():
    for tag, m, th in [("default", r["test_metrics_default"], 0.5),
                       ("tuned",   r["test_metrics"], r["best_threshold"])]:
        per_class = f1_score(m["binary_true"], m["binary_pred"],
                             average=None, labels=[0, 1], zero_division=0)
        print(f"{name + ' (' + tag + ')':<26} {th:>6.2f}  "
              f"{m['binary_accuracy']:>9.4f}  {m['binary_macro_f1']:>8.4f}  "
              f"{per_class[0]:>8.4f}  {per_class[1]:>8.4f}")
    print()

## 9. Test-set majority-class baseline (matched to the actual test split)

In [ ]:
# =============================================================
# 9. TEST-SET MAJORITY-CLASS BASELINE
# =============================================================
train_majority_binary = Counter(train_df["binary_id"]).most_common(1)[0][0]

test_binary_true = test_df["binary_id"].values
baseline_test_binary_pred = [train_majority_binary] * len(test_df)
yes_idx = CONFIG["binary_classes"].index("yes")

baseline_row = {
    "model": "majority_baseline",
    "binary_accuracy": accuracy_score(test_binary_true, baseline_test_binary_pred),
    "binary_f1": f1_score(test_binary_true, baseline_test_binary_pred,
                          average="binary", pos_label=yes_idx, zero_division=0),
    "binary_macro_f1": f1_score(test_binary_true, baseline_test_binary_pred,
                                average="macro", labels=[0, 1], zero_division=0),
}
print(f"Majority class in TRAIN: '{CONFIG['binary_classes'][train_majority_binary]}'")
print(f"TEST split: n={len(test_df)}  "
      f"({dict(Counter(test_df['binary_label']))})")
print("\nMajority-class baseline on TEST split:")
print(json.dumps(baseline_row, indent=2))

## 10. Results summary — generic buying-intent vs. majority baseline

In [ ]:
# =============================================================
# 10. RESULTS SUMMARY
# =============================================================
summary_rows = [baseline_row]
for name, r in results.items():
    summary_rows.append({
        "model": name,
        "binary_accuracy": r["test_binary_accuracy"],
        "binary_f1": r["test_binary_f1"],
        "binary_macro_f1": r["test_binary_macro_f1"],
    })
summary_df = pd.DataFrame(summary_rows)
summary_df["macro_f1_over_baseline"] = (
    summary_df["binary_macro_f1"] - baseline_row["binary_macro_f1"]).round(4)
display(summary_df.round(4))

summary_df.to_csv(os.path.join(CONFIG["out_dir"],
                               f"results_summary_{CONFIG['run_tag']}.csv"), index=False)
print(f"\nSaved results_summary_{CONFIG['run_tag']}.csv")

### 10a. Bar chart — accuracy / macro-F1 across encoders + baseline

In [ ]:
plot_df = summary_df.melt(
    id_vars="model", value_vars=["binary_accuracy", "binary_macro_f1"],
    var_name="metric", value_name="score",
)
plt.figure(figsize=(9, 5))
sns.barplot(data=plot_df, x="metric", y="score", hue="model")
plt.ylim(0, 1)
plt.axhline(baseline_row["binary_macro_f1"], ls="--", c="grey", lw=1)
plt.title("Generic buying-intent: v14-antiphony vs. majority baseline (test split)")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"summary_bar_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

### 10b. Confusion matrices — per encoder

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(5.5 * len(results), 4.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r["test_metrics"]["binary_true"], r["test_metrics"]["binary_pred"],
                          labels=list(range(len(CONFIG["binary_classes"]))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CONFIG["binary_classes"], yticklabels=CONFIG["binary_classes"])
    ax.set_title(f"{name} -- buying_intent")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"confusion_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

### 10c. Per-class classification reports

In [ ]:
for name, r in results.items():
    print(f"\n{'='*70}\n{name} -- buying_intent classification report\n{'='*70}")
    print(classification_report(r["test_metrics"]["binary_true"],
                                r["test_metrics"]["binary_pred"],
                                labels=[0, 1], target_names=CONFIG["binary_classes"],
                                zero_division=0))

### 10d. Training curves — validation macro-F1 per epoch

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 4.5))
for name, r in results.items():
    hist = pd.DataFrame(r["history"])
    ax.plot(hist["epoch"], hist["val_binary_macro_f1"], marker="o", label=name)
ax.axhline(baseline_row["binary_macro_f1"], ls="--", c="grey", lw=1, label="majority baseline")
ax.set_xlabel("epoch"); ax.set_ylabel("val binary macro-F1")
ax.set_title("Validation macro-F1 (checkpoint selection metric)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], f"training_curves_{CONFIG['run_tag']}.png"), dpi=150)
plt.show()

## 11. Ablation — how much does the *dialogue* actually contribute?

**This is the section that determines what the headline number means.**

The generic labels are derived from explicit decision moves: a Buyer saying
*"lemme think, text u later"* is `DEFERRED_CONSIDERATION` → `no`, and
*"ok bhai, confirm my order"* is `PURCHASE_COMMITTED` → `yes`. If that single
turn alone is sufficient, then a high score from the dual-stream model
measures **label explicitness**, not dialogue understanding — and reporting it
as evidence of a working dialogue-understanding pipeline would overclaim.

Note from the §2b audit: only ~12 % of these dialogues end on a Buyer turn, so
"the last Buyer turn" is usually **mid-dialogue**, not the transcript's final
line — the merchant almost always closes with a courtesy. The control is
therefore a fair one, and this is also why the architecture reads the last
turn of the *Buyer stream* rather than the last turn overall.

The control isolates exactly that: the **same encoder**, the **same splits**,
the **same loss and selection rule**, fed **only the Buyer's final turn** —
no merchant turns, no history, no role streams, no cross-attention, no GRU.
Just `[CLS] <last buyer turn>` → linear head.

**Read the gap, not the scores:**

- **Large gap** (full ≫ last-turn) — the architecture is genuinely using
  discourse structure. The pipeline transfers.
- **Near-zero gap** — the corpus is solvable from the closing line; the full
  architecture is not being tested by this data. State it as a corpus
  limitation and, if pursuing it, build a harder variant (e.g. dialogues
  whose closing turn is ambiguous and whose outcome depends on earlier
  negotiation).

A near-zero gap is a **useful result**, not a failed run — it is the same
kind of data-path finding as the donation project's 256-token truncation
discovery, caught deliberately this time rather than after the fact.

In [ ]:
# =============================================================
# 11. LAST-TURN-ONLY ABLATION
#     Same encoder / splits / loss / selection rule, fed ONLY the
#     Buyer's final turn.
# =============================================================

def extract_last_buyer_turn(text):
    """Return the text of the LAST [Buyer] turn.

    Falls back to the last turn of any role if the dialogue somehow has no
    Buyer turn -- cannot happen on this corpus (verified in 2b), but the
    fallback keeps the control well-defined rather than silently empty."""
    utts = split_into_utterances(text, max_utterances=10 ** 6)
    buyer = [t for role, t in utts if role == ROLE_BUYER]
    if buyer:
        return buyer[-1]
    return utts[-1][1] if utts else ""


class LastTurnDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.encodings = tokenizer(
            [extract_last_buyer_turn(t) for t in dataframe["text"]],
            padding=True, truncation=True, max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(dataframe["binary_id"].tolist(), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"input_ids": self.encodings["input_ids"][i],
                "attention_mask": self.encodings["attention_mask"][i],
                "labels": self.labels[i]}


class LastTurnClassifier(nn.Module):
    """Deliberately minimal: encoder -> masked mean-pool -> linear head.
    No role streams, no cross-attention, no trajectory GRU. Every mechanism
    the antiphony architecture adds is precisely what is absent here."""
    def __init__(self, hf_id, n_binary, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(hf_id)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, n_binary)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return self.head(self.dropout(pooled))


def run_lastturn_ablation():
    hf_id = next(e["hf_id"] for e in CONFIG["encoders"]
                 if e["name"] == CONFIG["ablation_encoder"])
    print(f"{'='*70}\nABLATION: last Buyer turn only -- {hf_id}\n{'='*70}")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    bs = CONFIG["ablation_batch_size"]
    loaders = {
        "train": DataLoader(LastTurnDataset(train_df, tokenizer, CONFIG["ablation_max_len"]),
                            batch_size=bs, shuffle=True),
        "val":   DataLoader(LastTurnDataset(val_df, tokenizer, CONFIG["ablation_max_len"]),
                            batch_size=bs),
        "test":  DataLoader(LastTurnDataset(test_df, tokenizer, CONFIG["ablation_max_len"]),
                            batch_size=bs),
    }

    # Show what the control actually reads -- makes the ablation legible.
    print("\nSample inputs the ablation sees (last Buyer turn only):")
    for t, lab in list(zip(test_df["text"], test_df["binary_label"]))[:5]:
        print(f"  [{lab:>3}] {extract_last_buyer_turn(t)[:100]}")

    model = LastTurnClassifier(hf_id, len(CONFIG["binary_classes"])).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"],
                                  weight_decay=CONFIG["weight_decay"], eps=1e-6)
    total_steps = len(loaders["train"]) * CONFIG["ablation_epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * CONFIG["warmup_ratio"]), total_steps)

    def eval_flat(loader):
        model.eval()
        yt, yp = [], []
        with torch.no_grad():
            for b in loader:
                labels = b.pop("labels")
                logits = model(**{k: v.to(DEVICE) for k, v in b.items()})
                yp.extend(logits.argmax(-1).cpu().tolist())
                yt.extend(labels.tolist())
        return {"binary_true": yt, "binary_pred": yp,
                "binary_accuracy": accuracy_score(yt, yp),
                "binary_f1": f1_score(yt, yp, average="binary",
                                      pos_label=CONFIG["binary_classes"].index("yes"),
                                      zero_division=0),
                "binary_macro_f1": f1_score(yt, yp, average="macro", labels=[0, 1],
                                            zero_division=0)}

    ckpt = os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"], "ablation_lastturn.pt")
    best, history = -1.0, []
    # Same class-balanced focal objective as the full model, so the
    # comparison isolates the INPUT, not the loss.
    for epoch in range(1, CONFIG["ablation_epochs"] + 1):
        model.train()
        run = 0.0
        for b in tqdm(loaders["train"], desc=f"[ablation] epoch {epoch}/{CONFIG['ablation_epochs']}"):
            labels = b.pop("labels").to(DEVICE)
            optimizer.zero_grad()
            logits = model(**{k: v.to(DEVICE) for k, v in b.items()})
            loss = binary_focal_loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
            optimizer.step(); scheduler.step()
            run += loss.item()
        vm = eval_flat(loaders["val"])
        history.append({"epoch": epoch, "train_loss": run / len(loaders["train"]),
                        "val_binary_macro_f1": vm["binary_macro_f1"]})
        print(f"  epoch {epoch}: train_loss={history[-1]['train_loss']:.4f} "
              f"val_macroF1={vm['binary_macro_f1']:.4f}")
        if vm["binary_macro_f1"] > best:
            best = vm["binary_macro_f1"]
            torch.save(model.state_dict(), ckpt)
            print(f"  -> new best ({best:.4f}), checkpoint saved.")

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    tm = eval_flat(loaders["test"])
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\nAblation TEST: acc={tm['binary_accuracy']:.4f} "
          f"macroF1={tm['binary_macro_f1']:.4f}")
    return {"name": f"ablation_lastturn_{CONFIG['ablation_encoder']}",
            "history": history, "test_metrics": tm,
            "test_binary_accuracy": tm["binary_accuracy"],
            "test_binary_f1": tm["binary_f1"],
            "test_binary_macro_f1": tm["binary_macro_f1"]}


ablation_result = run_lastturn_ablation() if CONFIG.get("run_lastturn_ablation") else None

### 11a. The comparison that matters — full dialogue vs. closing turn alone

In [ ]:
# =============================================================
# 11a. FULL MODEL vs. LAST-TURN CONTROL
# =============================================================
if ablation_result is not None:
    best_full = max(results.values(), key=lambda r: r["test_binary_macro_f1"])
    abl_f1  = ablation_result["test_binary_macro_f1"]
    full_f1 = best_full["test_binary_macro_f1"]
    base_f1 = baseline_row["binary_macro_f1"]
    gap = full_f1 - abl_f1

    cmp_df = pd.DataFrame([
        {"system": "majority baseline", "input": "none",
         "accuracy": baseline_row["binary_accuracy"], "macro_f1": base_f1},
        {"system": f"last-turn control ({CONFIG['ablation_encoder']})",
         "input": "final Buyer turn only",
         "accuracy": ablation_result["test_binary_accuracy"], "macro_f1": abl_f1},
        {"system": f"v14-antiphony ({best_full['name']})", "input": "full dialogue, dual-stream",
         "accuracy": best_full["test_binary_accuracy"], "macro_f1": full_f1},
    ])
    display(cmp_df.round(4))

    print(f"\n{'='*70}")
    print(f"Dialogue contribution (full - last-turn) = {gap:+.4f} macro-F1")
    print(f"Last-turn lift over baseline             = {abl_f1 - base_f1:+.4f}")
    print(f"{'='*70}")
    if gap >= 0.05:
        print("READING: the full architecture is using discourse structure beyond the\n"
              "closing turn -- the dual-stream pipeline transfers to this domain as a\n"
              "DIALOGUE-understanding model, not just a last-utterance classifier.")
    elif gap >= 0.01:
        print("READING: a modest but real contribution from dialogue context. The closing\n"
              "turn carries most of the signal; report both numbers side by side rather\n"
              "than the full-model score alone.")
    else:
        print("READING: the closing turn alone is essentially sufficient. The pipeline runs\n"
              "correctly on this corpus, but this corpus does NOT test dialogue\n"
              "understanding -- the label is recoverable from one utterance. Report this\n"
              "as a corpus property; a harder variant would need dialogues whose outcome\n"
              "depends on earlier negotiation rather than an explicit closing move.")

    plt.figure(figsize=(7.5, 4.5))
    sns.barplot(data=cmp_df, x="macro_f1", y="system", orient="h")
    plt.xlim(0, 1); plt.xlabel("test macro-F1"); plt.ylabel("")
    plt.title("How much does the dialogue add over the closing turn?")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["out_dir"], f"ablation_{CONFIG['run_tag']}.png"), dpi=150)
    plt.show()
else:
    print("Ablation disabled (CONFIG['run_lastturn_ablation'] = False).")

## 12. Controlled-skew stress test — does the imbalance machinery transfer?

The donation corpus was **68.2 / 31.8** skewed; this one is **47.6 / 52.4**,
essentially balanced. That difference is a confound in the cross-corpus
comparison: if this corpus scores well, a reader can fairly object that the
architecture merely had an easier class distribution, and that the
class-balanced machinery carried over from v14 was never actually exercised.

This section removes that objection by **manufacturing the skew and measuring
what happens.** The training split is subsampled to progressively harsher
ratios — the natural ~48 % `yes`, then 30 %, 15 % and 10 % — and each ratio is
trained twice on the same subsample:

| arm | loss | selection | threshold |
|---|---|---|---|
| **protected** | class-balanced focal (Cui et al. 2019) + supervised contrastive | val macro-F1 | val-tuned |
| **unprotected** | plain cross-entropy | val macro-F1 | fixed 0.5 |

Everything else — architecture, encoder, seed, epochs, and the **untouched
val/test splits** — is held constant. Only the training distribution and the
imbalance handling vary, so the gap between the two curves is attributable to
the technique alone.

**Note on what each arm isolates.** The protected arm bundles all four
certified components. That is deliberate: the claim under test is "the
imbalance-handling *package* transfers", not the marginal contribution of each
part, which n=500 cannot resolve. A per-component ablation would need a much
larger corpus to separate.

**What to expect.** Class-balanced focal loss re-weights by *effective number
of samples*, `(1 − βⁿ)/(1 − β)`, which saturates gracefully as a class thins
out instead of exploding the way raw inverse-frequency weighting does. So the
protected curve should degrade slowly while the unprotected one falls off a
cliff — at 10 % `yes`, an unprotected model typically collapses toward
predicting the majority class for everything, which is exactly the 0.00-F1
failure mode the donation project hit repeatedly on its `conditional` class.

**Cost.** 4 ratios × 2 arms = 8 training runs on `roberta-base` only. This is
the most expensive section in the notebook — set
`CONFIG["run_skew_stress_test"] = False` to skip it, or trim
`CONFIG["skew_ratios"]`.

In [ ]:
# =============================================================
# 12. CONTROLLED-SKEW STRESS TEST
#     Same architecture, same val/test, same seed. Only (a) the train
#     class ratio and (b) whether the imbalance machinery is enabled.
# =============================================================

def subsample_to_ratio(frame, target_yes_frac, seed=SEED):
    """Downsample the MAJORITY-side rows so `yes` makes up target_yes_frac
    of the training split.

    Downsampling (never upsampling) keeps every retained row a real, unique
    dialogue -- so a drop in score is caused by class imbalance itself, not
    by duplicate rows inflating the effective epoch count."""
    yes_idx = CONFIG["binary_classes"].index("yes")
    yes_rows = frame[frame["binary_id"] == yes_idx]
    no_rows  = frame[frame["binary_id"] != yes_idx]

    # Keep all `no`, shrink `yes` to hit the ratio: n_yes = f/(1-f) * n_no
    n_yes_target = int(round(target_yes_frac / (1 - target_yes_frac) * len(no_rows)))
    n_yes_target = max(2, min(len(yes_rows), n_yes_target))   # >=2 for stratification
    yes_keep = yes_rows.sample(n=n_yes_target, random_state=seed)
    out = pd.concat([no_rows, yes_keep]).sample(frac=1.0, random_state=seed)
    return out.reset_index(drop=True)


class PlainCrossEntropy(nn.Module):
    """Unprotected control: no class weighting, no focal term. This is what
    the protected arm is being measured against."""
    def forward(self, logits, targets):
        return F.cross_entropy(logits, targets)


def train_skew_arm(train_subset, protected, tag, encoder_cfg):
    """One (ratio, arm) cell of the stress test.

    `protected` toggles all four certified components together:
      class-balanced focal loss, supervised contrastive auxiliary,
      and the val-tuned decision threshold. Macro-F1 checkpoint
      selection is kept in BOTH arms -- selecting the unprotected arm on
      accuracy instead would hand it a second, unrelated handicap and
      make the comparison unfair.
    """
    global binary_focal_loss_fn
    hf_id = encoder_cfg["hf_id"]

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    collate_fn = make_hier_collate_fn(tokenizer, CONFIG["max_utt_len"],
                                      CONFIG["max_utterances"])
    bs = encoder_cfg.get("batch_size", CONFIG["batch_size"])
    ebs = encoder_cfg.get("eval_batch_size", CONFIG["eval_batch_size"])

    train_loader = DataLoader(DialogueIntentDataset(train_subset), batch_size=bs,
                              shuffle=True, collate_fn=collate_fn)
    val_loader   = DataLoader(DialogueIntentDataset(val_df), batch_size=ebs,
                              shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(DialogueIntentDataset(test_df), batch_size=ebs,
                              shuffle=False, collate_fn=collate_fn)

    # Swap the module-level loss fn that compute_losses() closes over, so
    # the training path is otherwise byte-identical between the two arms.
    saved_loss_fn = binary_focal_loss_fn
    saved_contrastive = CONFIG["use_contrastive"]
    if protected:
        binary_focal_loss_fn = ClassBalancedFocalLoss(
            samples_per_class(train_subset["binary_id"].values,
                              len(CONFIG["binary_classes"])),
            beta=CONFIG["cb_beta"], gamma=CONFIG["focal_gamma"]).to(DEVICE)
    else:
        binary_focal_loss_fn = PlainCrossEntropy().to(DEVICE)
        CONFIG["use_contrastive"] = False

    try:
        torch.manual_seed(SEED)   # identical init across arms
        model = AntiphonyClassifier(
            hf_id, n_binary=len(CONFIG["binary_classes"]),
            stream_layers=CONFIG["antiphony_stream_layers"],
            stream_heads=CONFIG["antiphony_stream_heads"],
            cross_heads=CONFIG["antiphony_cross_heads"],
            max_role_len=CONFIG["antiphony_max_role_len"]).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=encoder_cfg.get("lr", CONFIG["lr"]),
                                      weight_decay=CONFIG["weight_decay"], eps=1e-6)
        n_epochs = CONFIG["skew_epochs"]
        total_steps = len(train_loader) * n_epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, int(total_steps * CONFIG["warmup_ratio"]), total_steps)

        ckpt = os.path.join(CONFIG["out_dir"], CONFIG["checkpoint_dir_name"],
                            f"skew_{tag}.pt")
        best = -1.0
        for epoch in range(1, n_epochs + 1):
            model.train()
            for batch in tqdm(train_loader, desc=f"[{tag}] epoch {epoch}/{n_epochs}",
                              leave=False):
                labels = batch.pop("binary_labels").to(DEVICE)
                batch.pop("modifier_labels")
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                optimizer.zero_grad()
                logits, shared = model(**batch)
                loss = compute_losses(logits, shared, labels, include_contrastive=True)
                if torch.isnan(loss):
                    optimizer.zero_grad(); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                optimizer.step(); scheduler.step()

            vm = evaluate(model, val_loader, DEVICE, binary_threshold=0.5)
            if vm["binary_macro_f1"] > best:
                best = vm["binary_macro_f1"]
                torch.save(model.state_dict(), ckpt)

        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))

        # Threshold tuning is part of the protected package.
        if protected:
            th, _ = find_optimal_binary_threshold(model, val_loader, DEVICE)
        else:
            th = 0.5
        tm = evaluate(model, test_loader, DEVICE, binary_threshold=th)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return tm, th
    finally:
        # Always restore, even if a run raises -- otherwise a failure here
        # would silently corrupt every later cell's loss function.
        binary_focal_loss_fn = saved_loss_fn
        CONFIG["use_contrastive"] = saved_contrastive


def run_skew_stress_test():
    enc_cfg = next(e for e in CONFIG["encoders"] if e["name"] == CONFIG["skew_encoder"])
    yes_idx = CONFIG["binary_classes"].index("yes")
    rows = []

    for ratio in CONFIG["skew_ratios"]:
        sub = subsample_to_ratio(train_df, ratio)
        actual = (sub["binary_id"] == yes_idx).mean()
        print(f"\n{'='*72}\nSKEW {ratio:.0%} yes  ->  n_train={len(sub)} "
              f"(actual yes={actual:.1%}, "
              f"yes={int((sub['binary_id']==yes_idx).sum())}, "
              f"no={int((sub['binary_id']!=yes_idx).sum())})\n{'='*72}")

        for protected in (True, False):
            arm = "protected" if protected else "unprotected"
            tag = f"{int(ratio*100)}_{arm}"
            tm, th = train_skew_arm(sub, protected, tag, enc_cfg)
            per_class = f1_score(tm["binary_true"], tm["binary_pred"],
                                 average=None, labels=[0, 1], zero_division=0)
            rows.append({"yes_frac": round(actual, 4), "n_train": len(sub),
                         "arm": arm, "threshold": th,
                         "accuracy": tm["binary_accuracy"],
                         "macro_f1": tm["binary_macro_f1"],
                         "no_f1": per_class[0], "yes_f1": per_class[1]})
            print(f"  {arm:<12} thresh={th:.2f}  acc={tm['binary_accuracy']:.4f}  "
                  f"macroF1={tm['binary_macro_f1']:.4f}  "
                  f"(no={per_class[0]:.3f}, yes={per_class[1]:.3f})")

    return pd.DataFrame(rows)


skew_df = run_skew_stress_test() if CONFIG.get("run_skew_stress_test") else None

### 12a. Skew stress test — results

In [ ]:
# =============================================================
# 12a. SKEW STRESS TEST RESULTS
# =============================================================
if skew_df is not None:
    display(skew_df.round(4))
    skew_df.to_csv(os.path.join(CONFIG["out_dir"],
                                f"skew_stress_{CONFIG['run_tag']}.csv"), index=False)

    pivot = skew_df.pivot(index="yes_frac", columns="arm", values="macro_f1")
    pivot["protection_gain"] = (pivot["protected"] - pivot["unprotected"]).round(4)
    print("\nmacro-F1 by train class ratio:")
    display(pivot.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for arm, style in [("protected", "-o"), ("unprotected", "--s")]:
        d = skew_df[skew_df["arm"] == arm].sort_values("yes_frac")
        axes[0].plot(d["yes_frac"], d["macro_f1"], style, label=arm)
        axes[1].plot(d["yes_frac"], d["yes_f1"], style, label=f"{arm} (yes F1)")
    axes[0].axhline(baseline_row["binary_macro_f1"], ls=":", c="grey",
                    label="majority baseline")
    axes[0].set_xlabel("fraction of 'yes' in TRAIN split")
    axes[0].set_ylabel("test macro-F1")
    axes[0].set_title("Imbalance robustness: macro-F1")
    axes[0].legend(); axes[0].set_ylim(0, 1); axes[0].invert_xaxis()
    axes[1].set_xlabel("fraction of 'yes' in TRAIN split")
    axes[1].set_ylabel("test F1 on the MINORITY ('yes') class")
    axes[1].set_title("Where collapse shows first: minority-class F1")
    axes[1].legend(); axes[1].set_ylim(0, 1); axes[1].invert_xaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG["out_dir"], f"skew_stress_{CONFIG['run_tag']}.png"), dpi=150)
    plt.show()

    worst = skew_df["yes_frac"].min()
    w_prot = skew_df[(skew_df.yes_frac == worst) & (skew_df.arm == "protected")].iloc[0]
    w_unprot = skew_df[(skew_df.yes_frac == worst) & (skew_df.arm == "unprotected")].iloc[0]
    gain = w_prot["macro_f1"] - w_unprot["macro_f1"]

    print(f"\n{'='*72}")
    print(f"At the harshest ratio tested ({worst:.1%} yes in train):")
    print(f"  protected   macro-F1 = {w_prot['macro_f1']:.4f}  "
          f"(minority 'yes' F1 = {w_prot['yes_f1']:.4f})")
    print(f"  unprotected macro-F1 = {w_unprot['macro_f1']:.4f}  "
          f"(minority 'yes' F1 = {w_unprot['yes_f1']:.4f})")
    print(f"  protection gain      = {gain:+.4f} macro-F1")
    print(f"{'='*72}")
    if w_unprot["yes_f1"] < 0.05:
        print("The unprotected arm COLLAPSED on the minority class (F1 < 0.05) -- it is\n"
              "predicting the majority class for nearly everything. This is the same\n"
              "failure mode the donation project hit on its `conditional` class, and the\n"
              "class-balanced machinery is what prevents it here.")
    elif gain >= 0.05:
        print("The certified imbalance handling gives a clear, material advantage under\n"
              "skew -- it transfers to this domain as an effective technique, not just\n"
              "inherited configuration.")
    else:
        print("Both arms held up at this skew level. Either the task is easy enough that\n"
              "imbalance does not bite yet, or harsher ratios are needed to separate them --\n"
              "extend CONFIG['skew_ratios'] downward (e.g. 0.05) to find the breaking point.")
else:
    print("Skew stress test disabled (CONFIG['run_skew_stress_test'] = False).")

## 13. Slice-wise error analysis

Where the model fails is more informative than the aggregate score.
`outcome_category` and `archetype` were never training targets, so these
slices are a genuinely held-out view of *which* failure mode the model
confuses — e.g. is `DEFERRED_CONSIDERATION` (a soft "no") being read as a
`yes`? That is the generic-domain analogue of the donation corpus's
conditional/deferred boundary, the same boundary its annotators disagreed on.

In [ ]:
# =============================================================
# 12. SLICE-WISE ERROR ANALYSIS  (analysis columns, never trained on)
# =============================================================
best_name = max(results, key=lambda n: results[n]["test_binary_macro_f1"])
best_r = results[best_name]
print(f"Analysing best encoder: {best_name} "
      f"(test macro-F1={best_r['test_binary_macro_f1']:.4f})\n")

err_df = test_df.copy().reset_index(drop=True)
err_df["true"] = best_r["test_metrics"]["binary_true"]
err_df["pred"] = best_r["test_metrics"]["binary_pred"]
err_df["prob_yes"] = best_r["test_metrics"]["binary_probs"]
err_df["correct"] = err_df["true"] == err_df["pred"]

for col in ["outcome_category", "archetype", "category", "product"]:
    if col not in err_df.columns:
        continue
    g = err_df.groupby(col).agg(n=("correct", "size"), accuracy=("correct", "mean"))
    g = g[g["n"] >= 3].sort_values("accuracy")
    if len(g):
        print(f"--- accuracy by {col} (slices with n>=3) ---")
        display(g.round(4))

# The most valuable artefact: the actual misclassified closing turns.
wrong = err_df[~err_df["correct"]]
print(f"\n{'='*78}\nMISCLASSIFIED DIALOGUES: {len(wrong)}/{len(err_df)}\n{'='*78}")
for _, row in wrong.head(15).iterrows():
    true_lab = CONFIG["binary_classes"][row["true"]]
    pred_lab = CONFIG["binary_classes"][row["pred"]]
    print(f"\n[{row.get('dialogue_id', '?')}] true={true_lab} pred={pred_lab} "
          f"P(yes)={row['prob_yes']:.3f}  outcome={row.get('outcome_category', '?')} "
          f"archetype={row.get('archetype', '?')}")
    print(f"  last Buyer turn: {extract_last_buyer_turn(row['text'])[:160]}")

err_df.drop(columns=["text"]).to_csv(
    os.path.join(CONFIG["out_dir"], f"test_predictions_{CONFIG['run_tag']}.csv"), index=False)
print(f"\nSaved test_predictions_{CONFIG['run_tag']}.csv")

## 14. Cross-corpus comparison & persisted artefacts

In [ ]:
# =============================================================
# 13. CROSS-CORPUS COMPARISON + SAVE ARTEFACTS
# =============================================================

# v14-antiphony on PersuasionForGood (test n=153, tuned threshold) --
# transcribed from ran-nb/ran-v14-antiphony.ipynb for reference only.
DONATION_V14 = {
    "roberta-base":    {"accuracy": 0.8889, "macro_f1": 0.8595},
    "deberta-v3-base": {"accuracy": 0.8954, "macro_f1": 0.8600},
    "todbert":         {"accuracy": 0.8758, "macro_f1": 0.8379},
    "majority":        {"accuracy": 0.7255, "macro_f1": 0.4205},
}

rows = []
for name in results:
    d = DONATION_V14.get(name)
    rows.append({
        "encoder": name,
        "donation_macroF1": d["macro_f1"] if d else np.nan,
        "donation_over_base": round(d["macro_f1"] - DONATION_V14["majority"]["macro_f1"], 4) if d else np.nan,
        "generic_macroF1": round(results[name]["test_binary_macro_f1"], 4),
        "generic_over_base": round(results[name]["test_binary_macro_f1"]
                                   - baseline_row["binary_macro_f1"], 4),
    })
cross_df = pd.DataFrame(rows)
print("Same architecture, two corpora. Compare the *_over_base columns --")
print("the raw macro-F1 columns are not comparable (different class balance).\n")
display(cross_df)

payload = {
    "run_tag": CONFIG["run_tag"],
    "corpus": "Generic e-commerce buying-intent (final.json -> Buying_Intent_Label.csv)",
    "n_total": int(len(df)),
    "n_train": int(len(train_df)), "n_val": int(len(val_df)), "n_test": int(len(test_df)),
    "augmentation": bool(CONFIG["use_augmentation"]),
    "baseline": baseline_row,
    "results": {n: {"test_binary_accuracy": r["test_binary_accuracy"],
                    "test_binary_f1": r["test_binary_f1"],
                    "test_binary_macro_f1": r["test_binary_macro_f1"],
                    "best_threshold": r["best_threshold"],
                    "history": r["history"]} for n, r in results.items()},
    "ablation": ({"test_binary_accuracy": ablation_result["test_binary_accuracy"],
                  "test_binary_macro_f1": ablation_result["test_binary_macro_f1"]}
                 if ablation_result else None),
    "cross_corpus": cross_df.to_dict(orient="records"),
}
out_json = os.path.join(CONFIG["out_dir"], f"run_report_{CONFIG['run_tag']}.json")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
print(f"\nSaved {out_json}")
print("\nArtefacts written to", CONFIG["out_dir"])
for f_ in sorted(os.listdir(CONFIG["out_dir"])):
    if CONFIG["run_tag"] in f_:
        print("  -", f_)

## 15. What to write up

Fill these in from the run above:

1. **Does the architecture transfer?** Compare `generic_over_base` against
   `donation_over_base` in §14. Similar margins over each corpus's own
   baseline ⇒ the dual-stream mechanism is a general dialogue-understanding
   component, not a donation-corpus artefact.
2. **Does the dialogue matter?** The §11a gap is the headline. Report the
   full-model score and the last-turn control **together** — the full score
   alone is not interpretable without it.
3. **Which encoder wins, and does the ordering hold?** On donation,
   DeBERTa-v3 ≈ RoBERTa > TOD-BERT. If that ordering survives a complete
   domain change, it says something about the encoders; if it inverts
   (TOD-BERT is dialogue-pretrained, and this is a more conventional
   task-oriented dialogue than donation solicitation), that is worth a
   sentence of its own.
4. **Does the imbalance handling transfer?** From §12a: report the
   protected-vs-unprotected macro-F1 curves and the protection gain at the
   harshest ratio. This is what answers the "the balanced corpus just made it
   easier" objection — it shows the class-balanced machinery holding the model
   up under a skew *harsher than the donation corpus's own*, rather than
   asserting the technique carried over because the code did.
5. **Where does it fail?** From §13: if errors concentrate in
   `DEFERRED_CONSIDERATION`, the model is confusing a soft "no" with a
   "yes" — the same commitment/deferral boundary that produced the donation
   annotators' disagreements, reappearing in a different domain. That
   recurrence is a finding, not a bug.

**Two caveats to state, not bury:** the corpus is largely LLM-generated
dialogue, so it measures transfer to *synthetic* dialogue rather than to
scraped human conversation; and at n=500 with a 75-row test split, differences
under roughly ±0.05 macro-F1 between encoders are not separable from noise.